# Statystyka dla Data Analyst/Engineer — Moduł 10: ANOVA

W Module 8 test t porównywał średnie DWÓCH grup. Co jednak, gdy grup jest **więcej niż
dwie** — jak nasze cztery regiony? W tym module sformalizujemy różnicę satysfakcji między
regionami, którą zobaczyliśmy wizualnie już w Module 2 (boxplot) — czas na formalny test.

## Spis treści
1. [Dlaczego nie wiele testów t naraz?](#sec1)
2. [Logika ANOVA: wariancja między grupami vs wewnątrz grup](#sec2)
3. [Jednoczynnikowa ANOVA w praktyce](#sec3)
4. [Założenia ANOVA](#sec4)
5. [Test post-hoc: Tukey HSD](#sec5)
6. [Wielkość efektu: eta kwadrat](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Dlaczego nie wiele testów t naraz?

Mając 4 regiony, moglibyśmy pomyśleć: po prostu zróbmy test t dla każdej pary regionów
osobno (Północ-Południe, Północ-Wschód, Północ-Zachód, Południe-Wschód, ...) — dla 4 grup
to aż `6` par. Problem: każdy pojedynczy test t ma `alpha=0.05` ryzyka błędu I rodzaju
(Moduł 7), ale przy 6 NIEZALEŻNYCH testach naraz, prawdopodobieństwo, że PRZYNAJMNIEJ
JEDEN z nich fałszywie wyjdzie istotny, rośnie znacznie powyżej 5%. To tzw. **problem
wielokrotnego porównywania** (*multiple comparisons problem*).

In [ ]:
alpha = 0.05
liczba_testow = 6
# prawdopodobienstwo co najmniej jednego falszywego alarmu przy 6 niezaleznych testach
p_co_najmniej_jeden_falszywy = 1 - (1 - alpha) ** liczba_testow
print(f"Ryzyko co najmniej jednego falszywego alarmu: {p_co_najmniej_jeden_falszywy:.3f}")


Prawie 27% zamiast zakładanych 5%! **ANOVA** (*Analysis of Variance*) rozwiązuje ten
problem, testując WSZYSTKIE grupy JEDNYM testem, z jedną, kontrolowaną wartością `alpha`.

<a id="sec2"></a>
## 2. Logika ANOVA: wariancja między grupami vs wewnątrz grup

Nazwa "analiza wariancji" brzmi myląco — ANOVA testuje różnice ŚREDNICH, ale robi to,
porównując DWA źródła wariancji:

- **Wariancja między grupami** (*between-group*) — jak bardzo różnią się od siebie
  średnie poszczególnych grup.
- **Wariancja wewnątrz grup** (*within-group*) — jak bardzo rozrzucone są obserwacje
  wewnątrz każdej pojedynczej grupy.

Intuicja: jeśli grupy naprawdę się różnią, wariancja MIĘDZY grupami powinna być duża w
porównaniu z "naturalnym szumem" WEWNĄTRZ grup. Statystyka testowa `F` to właśnie ten
stosunek:

```
F = wariancja miedzy grupami / wariancja wewnatrz grup
```

`F` bliskie `1` sugeruje brak różnicy (obie wariancje podobnej wielkości), `F` znacznie
większe od `1` sugeruje prawdziwą różnicę między średnimi grup.

- H0: wszystkie średnie grup są równe (`mu_1 = mu_2 = mu_3 = mu_4`)
- H1: przynajmniej jedna średnia grupy różni się od pozostałych

<a id="sec3"></a>
## 3. Jednoczynnikowa ANOVA w praktyce

Sprawdźmy formalnie różnicę satysfakcji między regionami, którą widzieliśmy na wykresie
pudełkowym w Module 2:

In [ ]:
from scipy import stats

grupy_regionow = [dane["satysfakcja"].values for _, dane in klienci.groupby("region")]

statystyka_f, wartosc_p = stats.f_oneway(*grupy_regionow)
print(f"F = {statystyka_f:.3f}")
print(f"wartosc p = {wartosc_p:.6f}")

alpha = 0.05
print("Odrzucamy H0" if wartosc_p <= alpha else "Nie odrzucamy H0")


Ten sam wynik, tylko z pełniejszym raportem (m.in. tabela ANOVA), można dostać przez
`statsmodels` z użyciem formuł (podobnie jak w regresji z Modułu 12):

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols("satysfakcja ~ C(region)", data=klienci).fit()
tabela_anova = sm.stats.anova_lm(model, typ=2)
tabela_anova


> 📊 **C(region) oznacza 'traktuj jako kategorię'**
>
> `C(...)` w formule `statsmodels` (*categorical*) mówi modelowi, żeby potraktował `region` jako zmienną kategoryczną (osobny efekt dla każdej wartości), a nie liczbę. Dla kolumn tekstowych jak `region` `statsmodels` zwykle domyśli się tego sam, ale jawne `C()` to dobry nawyk, szczególnie dla kolumn, które WYGLĄDAJĄ jak liczby (np. gdybyśmy zakodowali regiony jako 1/2/3/4).

<a id="sec4"></a>
## 4. Założenia ANOVA

Podobnie jak test t, ANOVA zakłada w przybliżeniu normalny rozkład wewnątrz każdej grupy
oraz **jednorodność wariancji** (*homogeneity of variance*) — że wariancja jest podobna we
wszystkich grupach. Sprawdźmy to samo narzędzie z Modułu 8 (Levene), tym razem dla więcej
niż dwóch grup naraz:

In [ ]:
print("Levene (wszystkie 4 regiony):", stats.levene(*grupy_regionow))


> ⚠️ **Gdy wariancje nie są jednorodne: Welch ANOVA**
>
> Klasyczna ANOVA (jak `f_oneway`) zakłada równe wariancje, podobnie jak klasyczny test t. Gdy Levene wskazuje na wyraźnie nierówne wariancje, odpowiednikiem testu Welcha z Modułu 8 jest **Welch ANOVA** (`pingouin.welch_anova` w bibliotece `pingouin`, spoza standardowego `scipy`/`statsmodels`) — bezpieczniejszy wybór przy wyraźnie różnej wariancji między grupami.

<a id="sec5"></a>
## 5. Test post-hoc: Tukey HSD

ANOVA odpowiada tylko na pytanie "czy PRZYNAJMNIEJ JEDNA grupa się różni?" — nie mówi
KTÓRE konkretnie pary grup różnią się od siebie. Do tego służą **testy post-hoc**
("po fakcie"), uruchamiane DOPIERO gdy ANOVA wyszła istotna. Najpopularniejszy to
**Tukey HSD** (*Honestly Significant Difference*) — porównuje wszystkie pary grup,
jednocześnie kontrolując ogólne ryzyko błędu I rodzaju (dokładnie ten problem z sekcji 1).

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

wynik_tukey = pairwise_tukeyhsd(
    endog=klienci["satysfakcja"], groups=klienci["region"], alpha=0.05
)
print(wynik_tukey)


Kolumna `reject` mówi wprost, dla której pary regionów różnica jest istotna
statystycznie (`True`) — to dokładnie ta informacja, której zwykła ANOVA nam nie dała, a
którą teraz mamy w bezpieczny, skorygowany sposób.

In [ ]:
fig = wynik_tukey.plot_simultaneous()


<a id="sec6"></a>
## 6. Wielkość efektu: eta kwadrat

**Eta kwadrat** (`eta^2`) to odpowiednik d Cohena (Moduł 8) i V Craméra (Moduł 9) dla
ANOVA — mierzy, jaki ODSETEK całkowitej wariancji satysfakcji da się wyjaśnić
przynależnością do regionu:

```
eta^2 = suma_kwadratow_miedzy_grupami / suma_kwadratow_calkowita
```

Umowne progi (podobne do d Cohena): `0.01` mały efekt, `0.06` średni, `0.14`+ duży.

In [ ]:
suma_kwadratow_miedzy = tabela_anova["sum_sq"]["C(region)"]
suma_kwadratow_calkowita = tabela_anova["sum_sq"].sum()
eta_kwadrat = suma_kwadratow_miedzy / suma_kwadratow_calkowita

print(f"eta^2 = {eta_kwadrat:.3f}")


<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- problem wielokrotnego porównywania jako powód, żeby NIE robić wielu testów t naraz,
- logikę ANOVA: porównanie wariancji między grupami i wewnątrz grup (statystyka F),
- jednoczynnikową ANOVA przez `scipy.stats.f_oneway` i `statsmodels`,
- założenia ANOVA i test Levene'a dla wielu grup,
- test post-hoc Tukey HSD do znalezienia KTÓRE pary grup różnią się istotnie,
- eta kwadrat jako miarę wielkości efektu.

> 📝 **Ćwiczenie 1: ANOVA dla wydatków wg miasta**
>
> Wykonaj jednoczynnikową ANOVA sprawdzającą, czy `wydatki_miesieczne` różnią się istotnie między 5 miastami. Zinterpretuj wynik przy `alpha=0.05`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
grupy_miast = [dane["wydatki_miesieczne"].values for _, dane in klienci.groupby("miasto")]
f_miast, p_miast = stats.f_oneway(*grupy_miast)
print(f"F={f_miast:.3f}, p={p_miast:.4f}")
```

Wydatki nie zaleza w naszych danych od miasta (tylko od losowego rozkladu prawoskosnego), wiec spodziewaj sie braku istotnosci.
</details>

> 📝 **Ćwiczenie 2: Levene dla miast**
>
> Sprawdź test Levene'a dla wydatków wg miasta z ćwiczenia 1. Czy założenie jednorodności wariancji jest spełnione?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
print(stats.levene(*grupy_miast))
```
</details>

> 📝 **Ćwiczenie 3: Post-hoc dla wieku wg regionu**
>
> Wykonaj ANOVA dla `wiek` wg `region`, a jeśli wyjdzie istotna, uruchom Tukey HSD. Jeśli NIE wyjdzie istotna, napisz w komentarzu dlaczego uruchomienie post-hoc nie miałoby wtedy sensu.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
grupy_wieku = [dane["wiek"].values for _, dane in klienci.groupby("region")]
f_wieku, p_wieku = stats.f_oneway(*grupy_wieku)
print(f"F={f_wieku:.3f}, p={p_wieku:.4f}")
# wiek jest losowany calkowicie niezaleznie od regionu, wiec ANOVA prawdopodobnie
# NIE wyjdzie istotna -- uruchamianie Tukey HSD po nieistotnej ANOVA nie ma sensu,
# bo sama ANOVA juz mowi, ze nie ma dowodow na ZADNA roznice miedzy grupami
```
</details>

> 📝 **Ćwiczenie 4: Eta kwadrat dla wydatków wg miasta**
>
> Policz eta kwadrat dla modelu z ćwiczenia 1 (`wydatki_miesieczne ~ C(miasto)`, zbuduj tabelę ANOVA przez `statsmodels` jak w sekcji 3). Porównaj wielkość efektu z eta kwadrat dla regionu i satysfakcji z sekcji 6.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
model_miast = ols("wydatki_miesieczne ~ C(miasto)", data=klienci).fit()
tabela_miast = sm.stats.anova_lm(model_miast, typ=2)
eta2_miast = tabela_miast["sum_sq"]["C(miasto)"] / tabela_miast["sum_sq"].sum()
print(f"eta^2 (miasto): {eta2_miast:.3f}")
print(f"eta^2 (region, sekcja 6): {eta_kwadrat:.3f}")
```
</details>

> 🔥 **Wyzwanie: ANOVA dwuczynnikowa**
>
> Rozszerz model z sekcji 3 o DRUGI czynnik: `satysfakcja ~ C(region) + C(grupa_kampanii)`. Zbuduj tabelę ANOVA (`sm.stats.anova_lm(model, typ=2)`) i sprawdź wiersze dla obu czynników osobno. Czy `grupa_kampanii` wnosi istotny wkład w wyjaśnianie satysfakcji, oprócz regionu?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
model_dwuczynnikowy = ols("satysfakcja ~ C(region) + C(grupa_kampanii)", data=klienci).fit()
tabela_dwuczynnikowa = sm.stats.anova_lm(model_dwuczynnikowy, typ=2)
tabela_dwuczynnikowa
```

Wiersz C(region) powinien pozostac wyraznie istotny (tak jak w sekcji 3). Wiersz C(grupa_kampanii) moze zaskoczyc: satysfakcja w kodzie generujacym dane NIGDY nie zalezala od grupy kampanii (tylko od wydatkow i regionu) -- a mimo to jego wartosc p moze wyjsc ponizej 0.05. To nie blad w danych, tylko zywy przyklad bledu I rodzaju z Modulu 7: przy alpha=0.05 nawet PRAWDZIWIE zerowy efekt da falszywie istotny wynik mniej wiecej raz na 20 prob -- i ten konkretny, ustalony seed losowy akurat na to trafil.
</details>

## Co dalej?

ANOVA sprawdza RÓŻNICE między grupami. W **Module 11** zmienimy perspektywę na
**ZWIĄZEK** między dwiema zmiennymi ciągłymi: korelację i kowariancję — fundament pod
regresję liniową (Moduł 12), która pozwoli nam nie tylko stwierdzić związek, ale i
przewidywać jedną zmienną na podstawie drugiej.